In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import pandas as pd
import json
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import time
from markdown_it import MarkdownIt
import numpy as np
import enum
from collections import Counter

from pydantic import BaseModel, Field
from langchain_google_vertexai import ChatVertexAI
from langchain_google_vertexai import VertexAIEmbeddings

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')


In [ ]:
!pip install langchain_google_vertexai langgraph google-cloud-aiplatform "anthropic[vertex]" scikit-fuzzy

# Prepare

Read corresponding file with Reddit submissions (to get one, apply for Reddit Data API and get an API key):

In [ ]:
#subreddit, file_name, disease = "BRCA", "submissions", "breast cancer"
subreddit, file_name, disease = "kidneycancer", "submissions_kc", "kidney cancer"
#subreddit, file_name, disease = "ProstateCancer", "submissions_pc", "prostate cancer"

with open(f"{file_name}.json", "r") as rf:
  submissions = json.loads(rf.read())["submissions"]

if file_name == "submissions_kc":
  submissions = submissions[1:]

texts = [s["text"] for s in submissions]
titles = [s["title"] for s in submissions]

# Utils

In [ ]:
import asyncio
semaphore = asyncio.Semaphore(20)

count = 0

async def process_task(task, debug=False):

  global count
  async with semaphore:
    count += 1
    if debug and count % 20 == 0:
      print(count)
    return await task

# Clustering scratch

This is the key workflow that iterates over texts and clusters them:

In [ ]:
prompt_template_decide = PromptTemplate.from_template(
    "You're a medical expert in text classification of informational needs of patients with {disease}. "
    "Your task is to analyze a new text and determine if it belongs to one of the existing clusters based on their descriptions. "
    "If the new text does not fit into any of the existing clusters, you must classify it as a 'NEW' one"
    "\n\n** EXISTING CLUSTER DESCRIPTIONS: **\n{clusters}\n\n"
    "\n\n** NEW TEXT TO CLASSIFY: **\nTITLE:\n{title}\n\SUBMISSTION:\n{text}\n\n"
    "\n\n** INSTRUCTIONS: **\n\n"
    "1. **Analyze:** Carefully read the 'NEW TEXT TO CLASSIFY' and understand its main topic, content, and intent."
    "2. **Compare:** Compare the meaning and themes of the new text to the descriptions of each existing cluster."
    "3. **Classify:**:\n."
    "   * If the new text strongly aligns with the description of an existing cluster, assign it to that cluster.\n"
    "   * If the new text does not fit well into any of the existing clusters, assign it to 'NEW' cluster.\n"
    "4. **Prioritize assigning to an existing cluster:** Always try to look for an existing cluster first before deciding it's a new one. \n\n"
    "Now, assign the new text provided to you to a cluster."
)

prompt_template_merge = PromptTemplate.from_template(
    "You're a medical expert in text classification of informational needs of patients with {disease}. "
    "\n\n** CLUSTER DESCRIPTION: **\n{cluster}\n\n"
    "\n\n** NEW TEXT: **\nTITLE:\n{title}\nSUBMISSTION:\n{text}\n\n"
    "Your task is to update an existing cluster description to incorporate patient's informational needs from a new text that has been assigned to this cluster. "
    "The goal is to make the description more comprehensive and representative of all its members, including the new text, without losing the core identity of the cluster."
    "\n\n** INSTRUCTIONS: **\n\n"
    "1. **Analyze the Core Theme:** First, carefully read the 'CLUSTER DESCTIPTION' to understand the central topic and boundaries of the cluster.\n"
    "2. **Analyze the New Information:** Next, analyze the 'New Text' to identify its key information, topics, and nuances.\n"
    "3. **Synthesize and Update:** Rewrite the 'CLUSTER DESCTIPTION' by integrating the key aspects of the 'New Text'. The updated description should be a "
    "concise and accurate summary of the cluster's content, now including the themes from the new text."
    "4. **Maintain Generality:** Do NOT make the description overly specific to the new text. Instead, generalize from the new text to broaden the existing "
    "description appropriately. The description should remain a high-level summary.\n"
    "5. **Preserve Clarity:** Ensure the updated description is clear, concise, and easy to understand.\n"
    "6. **Do not assume anything: ** Use only the patient's question provided below.\n"
    "7. **Be concise: ** Do not add additional details or describe the context. Capture ALL informational needs from both patients but be concise.\n"
    "8. **Text only: ** Do not use lists or bullet points. \n"
    "9. **Jump right to the point: ** Start immediately with describing information needs and be concise.\n"
    "Now, write a new cluster description based on the existing one and the new text."
)

In [ ]:
from langchain_google_vertexai.callbacks import VertexAICallbackHandler
tc_callback_gemini = VertexAICallbackHandler()

llm_ht = ChatVertexAI(
    model="gemini-2.5-pro", endpoint_version="v1",
    location="global", temperature=1., max_retries=100,
    callbacks=[tc_callback_gemini])
llm_lt = ChatVertexAI(
    model="gemini-2.5-pro", endpoint_version="v1",
    location="global", temperature=0., max_retries=100,
    callbacks=[tc_callback_gemini])

def format_final_summaries(cluster_descriptions):
  formatted = [f"CLUSTER_{i+1}: {s}" for i, s in enumerate(cluster_descriptions)]
  return "\n".join(formatted)

chain_merge = prompt_template_merge | llm_lt | StrOutputParser()

In [ ]:
from enum import Enum

p = 32
clusters_amnt = []
clusters = []
final_clusters = []
i = 0

In [ ]:
import random

sampled_submissions = random.choices(submissions, k=len(submissions))

In [ ]:
sampled_indices = []
for s in sampled_submissions:
  found = False
  for ind, s1 in enumerate(submissions):
    if s["text"] == s1["text"] and s["title"] == s1["title"]:
      sampled_indices.append(ind)
      found = True
      continue
  if not found:
    print(s)

In [ ]:
start = i
raw_summary_ids = []

for i, submission in enumerate(sampled_submissions[start:]):

  allowed_values = [f"CLUSTER_{k+1}" for k in range(len(final_clusters))] + ["NEW"]

  ClusterEnum = Enum( "ClusterEnum", {value.upper(): value for value in allowed_values})
  class Cluster(BaseModel):
    cluster_id: ClusterEnum = Field(description="Cluster representation that describe patient's informational need")

  summary_ids = []
  chain_decide = prompt_template_decide | llm_ht.with_structured_output(Cluster)
  tasks = []

  formatted_clusters = format_final_summaries(final_clusters)

  async def run(retry=1, max_retries = 10):
    try:
      return await chain_decide.ainvoke({
          "title": submission["title"], "text": submission["text"], "disease": disease, "clusters": formatted_clusters})
    except Exception as e:
      print(e)
      if retry < max_retries:
        return await run(retry+1)

  tasks = [process_task(run()) for _ in range(p)]
  results = await asyncio.gather(*tasks)
  for result in results:
    if not result:
      continue
    cluster_id = result.cluster_id
    if cluster_id == ClusterEnum.NEW:
      summary_ids.append(len(final_clusters))
    else:
      summary_ids.append(int(cluster_id.value.split("_")[-1])-1)

  raw_summary_ids.append(summary_ids)
  counts = Counter(summary_ids)
  clusters.append(summary_ids)
  summary_id = counts.most_common(1)[0][0]


  cluster = final_clusters[summary_id] if summary_id < len(final_clusters) else ""
  rewrite_summary = chain_merge.invoke({
      "title": submission["title"], "text": submission["text"], "disease": disease, "cluster": cluster})
  if summary_id < len(final_clusters):
    final_clusters[summary_id] = rewrite_summary
  else:
    final_clusters.append(rewrite_summary)
  clusters_amnt.append(len(final_clusters))

  if i % 10 == 0:
    print(i+start, len(final_clusters))
    with open(f"{file_name}_clusters_gem_25_0.json", "w") as wf:
      wf.write(json.dumps({"clusters": final_clusters, "clusters_amnt": clusters_amnt, "cluster_ids": clusters, "indices": sampled_indices}))

with open(f"{file_name}_clusters_gem_25_0.json", "w") as wf:
  wf.write(json.dumps({"clusters": final_clusters, "clusters_amnt": clusters_amnt, "cluster_ids": clusters, "indices": sampled_indices}))

# Agreement

Now, let's compute agreement rates:

In [ ]:
subreddits = ["BRCA", "kidneycancer", "ProstateCancer"]
file_names = ["submissions", "submissions_kc", "submissions_pc"]
diseases = ["breast cancer", "kidney cancer", "prostate cancer"]

results = []
for sr, fn, d in zip(subreddits, file_names, diseases):
  with open(f"{fn}.json", "r") as rf:
    submissions = json.loads(rf.read())["submissions"]
  if fn == "submissions_kc":
    submissions = submissions[1:]

  with open(f"{fn}_clusters_gem_25f_0.json", "r") as rf:
    data_temp = json.loads(rf.read())
    results.append(data_temp)


In [ ]:
def calculate_agreement(cluster_ids):
  agreement = []
  p = 32
  for i in range(p-3):
    same = 0
    for el in cluster_ids:
      summary_id = Counter(el[:(i+3)]).most_common(1)[0][0]
      max_summary_id = Counter(el).most_common(1)[0][0]
      if summary_id == max_summary_id:
        same += 1
    agreement.append(same/len(cluster_ids))
  return agreement

In [ ]:
agreement = [calculate_agreement(r["cluster_ids"]) for r in results]

In [ ]:
for sr, agr in zip(subreddits, agreement):
  plt.plot(range(1, len(agr)+1), agr, label=f"{sr}")
plt.legend()
plt.xlabel("Sampling amount, p")
plt.ylabel("Agreement rate")
plt.show()

In [ ]:
for sr, agr in zip(subreddits, agreement):
  plt.plot(range(1, len(agr)+1), agr, label=f"{sr}")
plt.legend()
plt.xlabel("Sampling amount, p")
plt.ylabel("Agreement rate")
plt.show()

In [ ]:
for sr, r in zip(subreddits, results):
  clusters_amnt = r["clusters_amnt"]
  plt.plot(range(1, len(clusters_amnt)+1), clusters_amnt, label=f"{sr}")
plt.legend()
plt.xlabel("Submissions processed")
plt.ylabel("Amount of clusters")
plt.show()

In [ ]:
for sr, r in zip(subreddits, results):
  clusters_amnt = r["clusters_amnt"]
  plt.plot(range(1, len(clusters_amnt)+1), clusters_amnt, label=f"{sr}")
plt.legend()
plt.xlabel("Submissions processed")
plt.ylabel("Amount of clusters")
plt.show()

# Interpretable clusters

We cann add additional interpretability step:

In [ ]:
llm = ChatVertexAI(
    model_name="gemini-2.5-pro", temperature=0., max_retries=10)

In [ ]:
prompt_finalize = PromptTemplate.from_template(
    "Summarize informational needs described below in 2-5 words. "
    "Do not assume anything, use only information provided to you. "
    "\nINFORMATIONAL NEEDS:\n{need}"
)
chain_cluster_header = prompt_finalize | llm | StrOutputParser()


def format_submissions(submissions_slice):
  data = [f"QUESTION TITLE:\n{s['title']}\nQUESTION:\n{s['text']}\n" for s in submissions_slice]
  return "\n".join(data)

prompt_template_cluster_description = PromptTemplate.from_template(
  "You're a medical experty studying information needs of patients with {disease}. "
  "Your task is to provide a short and concise summary based on questions from patiens and their relatives submitted on Reddit. "
  "\n\nINSTRUCTIONS:\n\n"
  "  1. Be concise. Provide only a short summary that is generalizable and useful for a practitioner.\n"
  "  2. Do not assume anything, use only the data provided to you.\n"
  "  3. Jump right to the point, start immediately with describing information needs. Do not add any additional descriptions.\n"
  "\n\nQUESTIONS:\n{questions}\n"
  "Now, provide a short and concise summary of their needs."
)

chain_cluster_description = prompt_template_cluster_description | llm | StrOutputParser()


In [ ]:
final_consensus = []
for result in cluster_ids:
  counts = Counter(result)
  final_consensus.append(counts.most_common(1)[0][0])

In [ ]:
def get_cluster_sizes(result):
  clusters = {}
  for cluster_id in result:
    if cluster_id not in clusters:
      clusters[cluster_id] = 0
    clusters[cluster_id] += 1
  return clusters

In [ ]:
cluster_sizes = get_cluster_sizes(final_consensus)

In [ ]:
sampled_indices = []
for s in submissions:
  found = False
  for ind, s1 in enumerate(submissions):
    if s["text"] == s1["text"] and s["title"] == s1["title"]:
      sampled_indices.append(ind)
      found = True
      continue
  if not found:
    print(s)

In [ ]:
import asyncio
import time

tasks = []
for label in cluster_sizes.keys():
  submissions_slice = [s for s, f in zip(submissions, final_consensus) if f == label]
  tasks.append(chain_cluster_description.ainvoke({"questions": format_submissions(submissions_slice), "disease": disease}))
  time.sleep(1)

clusters_descriptions = await asyncio.gather(*tasks)

In [ ]:
tasks = [chain_cluster_header.ainvoke(s) for s in clusters_descriptions]
topics = await asyncio.gather(*tasks)

In [ ]:
with open(f"{file_name}_clusters_gem_25f_0.json", "w") as wf:
  wf.write(json.dumps(
      {"clusters": final_clusters,
       "clusters_amnt": clusters_amnt,
       "cluster_ids": cluster_ids,
       "sampled_indices": sampled_indices,
       "topics": topics,
       "clusters_descriptions": clusters_descriptions,
       "labels": final_consensus
      }))

# HTML

In [ ]:
def render_cluster_snippet(title, text, cluster_id):
  md = MarkdownIt('commonmark', {'breaks':True,'html':True})
  html_text = md.render(text)
  template = (
      f"<div class=\"cluster-subheading\" onclick=\"toggleCluster(event, '{cluster_id}')\">{title}</div>"
      f"<div class=\"cluster-content\" id=\"cluster-content-{cluster_id}\">"
      f"<div class=\"item-markdown\">{html_text}</div></div>"
      #f"{text}</div>"
  )
  return template


def render_snippets_block(subheading, cluster_snippets, block_id):
  return f"""
    <div class="main-clustering-subheading" onclick="toggleAllClusters('{block_id}')">
        {subheading}
    </div>

    <div class="all-clusters-container" id={block_id}>
        {''.join(cluster_snippets)}
    </div>
  """

def generate_html_template(blocks):
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
    <title>Expandable Clusters</title>
    <style>
    .main-clustering-subheading {{
        cursor: pointer;
        padding: 10px;
        background-color: #ddd;
        border: 1px solid #bbb;
        margin-bottom: 10px;
    }}

    .all-clusters-container {{
        display: none;
        padding: 15px;
        border: 1px solid #aaa;
        margin-bottom: 20px;
        background-color: #f9f9f9;
    }}

    .cluster-subheading {{
        cursor: pointer;
        padding: 8px;
        background-color: #eee;
        border: 1px solid #ccc;
        margin-bottom: 3px;
    }}

    .cluster-content {{
        display: none;
        padding: 10px;
        border: 1px solid #ccc;
        margin-bottom: 5px;
        background-color: #fff;
    }}

    .item-title {{
      font-weight: bold;
      margin-bottom: 3px;
    }}
    .item-markdown {{
      margin-bottom: 8px;
      white-space: pre-wrap; /* Preserve whitespace and line breaks */
    }}
    </style>
    </head>
    <body>
    <h1>Clusters</h1>
    {''.join(blocks)}

    <script>
    function toggleAllClusters(elementId) {{
        var container = document.getElementById(elementId);
        if (container.style.display === "block") {{
            container.style.display = "none";
        }} else {{
            container.style.display = "block";
        }}
    }}

    function toggleCluster(event, clusterId) {{
        if (event) {{
            event.stopPropagation();
        }}
        var content = document.getElementById("cluster-content-" + clusterId);
        if (content.style.display === "block") {{
            content.style.display = "none";
        }} else {{
            content.style.display = "block";
        }}
    }}
    </script>

    </body>
    </html>
    """
    return html_content


In [ ]:
def get_cluster_sizes(labels):
  cluster_sizes = {}
  for c in labels:
    if c not in cluster_sizes:
      cluster_sizes[c] = 0
    cluster_sizes[c] += 1
  return cluster_sizes

In [ ]:
with open(f"{file_name}_clusters_gem_25_0.json", "r") as rf:
  data = json.loads(rf.read())
  clusters_descriptions = data["clusters_descriptions"]
  topics = data["topics"]
  labels = []
  for result in data["cluster_ids"]:
    counts = Counter(result)
    labels.append(counts.most_common(1)[0][0])

cluster_sizes = get_cluster_sizes(labels)
cluster_snippets = []
for i, (topic, text) in enumerate(zip(topics, clusters_descriptions)):
    size = cluster_sizes[i]
    snippet = render_cluster_snippet(f"{topic} ({size} elements)", text, f"cluster-llm-{i}")
    cluster_snippets.append(snippet)

block = render_snippets_block("LLM-based clustering", cluster_snippets, "llm")
blocks = [block]


with open(f"{file_name}_clusters_gem_25f_0.json", "r") as rf:
  data = json.loads(rf.read())
  clusters_descriptions = data["clusters_descriptions"]
  topics = data["topics"]
  labels = []
  for result in data["cluster_ids"]:
    counts = Counter(result)
    labels.append(counts.most_common(1)[0][0])

cluster_sizes = get_cluster_sizes(labels)
cluster_snippets = []
for i, (topic, text) in enumerate(zip(topics, clusters_descriptions)):
    size = cluster_sizes[i]
    snippet = render_cluster_snippet(f"{topic} ({size} elements)", text, f"cluster-small-llm-{i}")
    cluster_snippets.append(snippet)

block = render_snippets_block("Small LLM-based clustering", cluster_snippets, "small-llm")
blocks.append(block)

cluster_sizes = get_cluster_sizes(labels)
cluster_snippets = []
for i, (topic, text) in enumerate(zip(topics, clusters_descriptions)):
    size = cluster_sizes[i]
    snippet = render_cluster_snippet(f"{topic} ({size} elements)", text, f"cluster-small-llm-{i}")
    cluster_snippets.append(snippet)

#block = render_snippets_block("Small LLM-based clustering", cluster_snippets, "small-llm")
#blocks.append(block)


with open(f"{file_name}_full_clusters.json", "r") as rf:
  clusters = json.loads(rf.read())


for k, v in clusters.items():
  for k1, v1 in v.items():
    heading = f"{k} clustering based on {k1}"
    cluster_sizes = get_cluster_sizes(v1["labels"])
    cluster_snippets = []
    for i, (topic, text) in enumerate(zip(v1["cluster_topics"], v1["cluster_descriptions"])):
      if i+1 in cluster_sizes:
        size = cluster_sizes[i+1]
        snippet = render_cluster_snippet(f"{topic} ({size} elements)", text, f"{k}-{k1}-{i}")
        cluster_snippets.append(snippet)
    block = render_snippets_block(heading, cluster_snippets, f"{k}-{k1}")
    blocks.append(block)

final_html = generate_html_template(blocks)
from IPython.display import HTML
HTML(final_html)

In [ ]:
with open(f"{file_name}.html", "w", encoding="utf-8") as f:
    f.write(final_html)

# Reducing clusters

We can reduce the number of clusters further by handling small clusters separately:

In [ ]:
with open(f"{file_name}_clusters_gem_25f_0.json", "r") as rf:
  data = json.loads(rf.read())
  final_clusters = data["clusters"]
  clusters_amnt = data["clusters_amnt"]
  cluster_ids = data["cluster_ids"]
  if "sampled_indices" in data:
    indices = data["sampled_indices"]
  else:
    indices = data["indices"]

assert len(cluster_ids) == len(submissions)

In [ ]:
cluster_ids = cluster_ids[1:]
clusters_amnt = clusters_amnt[1:]

In [ ]:
labels = []
for result in cluster_ids:
  counts = Counter(result)
  labels.append(counts.most_common(1)[0][0])

In [ ]:
cluster_sizes = get_cluster_sizes(labels)

In [ ]:
threshold = 10
allowed_clusters = []
for cluster_id, cluster_amounts in cluster_sizes.items():
  if cluster_amounts >= threshold:
    allowed_clusters.append(cluster_id)

allowed_values = [f"CLUSTER_{k+1}" for k in allowed_clusters]

ClusterEnum = Enum( "ClusterEnum", {value.upper(): value for value in allowed_values})

class Cluster(BaseModel):
  cluster_id: ClusterEnum = Field(description="Cluster representation that describe patient's informational need")

chain_decide = prompt_template_decide | llm_ht.with_structured_output(Cluster)


for cluster_id, cluster_amounts in cluster_sizes.items():
  if cluster_amounts < threshold:
    print(cluster_id, cluster_amounts)

In [ ]:
sampled_submissions = [submissions[i] for i in sampled_indices]

In [ ]:
formatted_clusters = format_final_summaries([c for i, c in enumerate(final_clusters) if i in allowed_clusters])

In [ ]:
p = 16
new_cluster_ids = []

for label, submission, ids in zip(labels, sampled_submissions, cluster_ids):
  if cluster_sizes[label] > threshold:
    new_cluster_ids.append(ids)
    continue
  summary_ids = []

  async def run(retry=1, max_retries = 10):
    try:
      return await chain_decide.ainvoke({
          "title": submission["title"], "text": submission["text"], "disease": disease, "clusters": formatted_clusters})
    except Exception as e:
      print(e)
      if retry < max_retries:
        return await run(retry+1)

  tasks = [process_task(run()) for _ in range(p)]
  results = await asyncio.gather(*tasks)
  for result in results:
    if not result:
      continue
    cluster_id = result.cluster_id
    summary_ids.append(int(cluster_id.value.split("_")[-1])-1)

  new_cluster_ids.append(summary_ids)
  print("done!", len(summary_ids))

In [ ]:
sampled_indices = []
for s in submissions:
  found = False
  for ind, s1 in enumerate(submissions):
    if s["text"] == s1["text"] and s["title"] == s1["title"]:
      sampled_indices.append(ind)
      found = True
      continue
  if not found:
    print(s)

In [ ]:
with open(f"{file_name}_clusters_gem_25_0_red.json", "w") as wf:
  wf.write(json.dumps({"clusters": final_clusters, "clusters_amnt": clusters_amnt, "cluster_ids": new_cluster_ids, "indices": indices}))


In [ ]:
cluster_names = list(cluster_sizes.keys())
cluster_amounts = [cluster_sizes[i] for i in cluster_ids]

def autopct_format(pct):
  return ('%.1f%%' % pct) if pct > 2 else ''

plt.figure(figsize=(10, 8))
labels_diag = [i if (t / sum(cluster_amounts)) > 0.02 else "" for i, t in zip(cluster_names, cluster_amounts)]
plt.pie(cluster_amounts, labels=labels_diag, autopct=autopct_format, startangle=90, pctdistance=0.85)
plt.tight_layout()

# Stability

In [ ]:
with open(f"{file_name}_clusters_gem_25.json", "r") as rf:
  data = json.loads(rf.read())

In [ ]:
bootstraps = []
for i in range(20):
  if i == 2 or i > 14:
    continue
  with open(f"{file_name}_clusters_gem_25f_{i}_red.json", "r") as rf:
    d = json.loads(rf.read())
    if "indices" not in d and "sampled_indices" in d:
      d["indices"] = d["sampled_indices"]
    if len(d["cluster_ids"]) == len(submissions) + 1:
      d["cluster_ids"] = d["cluster_ids"][1:]
    if "indices" in d:
      assert len(d["cluster_ids"]) == len(submissions)
      assert len(d["indices"]) == len(submissions)
      bootstraps.append(d)

In [ ]:
raw_cluster_ids = []
for el in data["cluster_ids"]:
  cluster_id = Counter(el).most_common(1)[0][0]
  raw_cluster_ids.append(cluster_id)

raw_cluster_ids = np.array(raw_cluster_ids)

In [ ]:
bootstrap_cluster_ids = []
bootstrap_cluster_ids_expected = []
p=16

for i, el in enumerate(bootstraps):
   bootstrap_ids = raw_cluster_ids[el["indices"]]
   bootstrap_cluster_ids_expected.append(bootstrap_ids)
   cluster_ids = []
   for e in el["cluster_ids"]:
    cluster_id = Counter(e[:p]).most_common(1)[0][0]
    cluster_ids.append(cluster_id)
   bootstrap_cluster_ids.append(cluster_ids)


In [ ]:
scores = []
n = len(bootstrap_cluster_ids_expected)
t_crit = stats.t.ppf((1 + 0.95) / 2, df=n-1)

for a, b in zip(bootstrap_cluster_ids, bootstrap_cluster_ids_expected):
  s = adjusted_rand_score(a, b)
  scores.append(s)

std_err = np.std(scores) / np.sqrt(n-1)
print(f"Score = {np.mean(scores)}, {t_crit * std_err}")

In [ ]:
from sklearn.metrics import adjusted_rand_score

def get_pairwise_stability(all_labels):
  n_runs = len(all_labels)
  scores = []
  for i in range(n_runs):
    for j in range(i + 1, n_runs):
      score = adjusted_rand_score(all_labels[i], all_labels[j])
      scores.append(score)

  return  np.mean(scores) if scores else 1.0

In [ ]:
el = bootstraps[3]
for e in el["cluster_ids"]:
  cluster_id = Counter(e[:p]).most_common(1)[0][0]


In [ ]:
bootstraps = []
for i in range(1, 20):
  with open(f"{file_name}_clusters_gem_25_0.json", "r") as rf:
    d = json.loads(rf.read())
    assert len(d["cluster_ids"]) == 822
    bootstraps.append(d)

In [ ]:
raw_cluster_ids = []
for el in data["cluster_ids"][1:]:
  cluster_id = Counter(el[:p]).most_common(1)[0][0]
  raw_cluster_ids.append(cluster_id)

raw_cluster_ids = np.array(raw_cluster_ids)

In [ ]:
bootstrap_cluster_ids = []
bootstrap_cluster_ids_expected = []

for el in bootstraps:
   bootstrap_ids = raw_cluster_ids[el["indices"]]
   bootstrap_cluster_ids_expected.append(bootstrap_ids)
   cluster_ids = []
   for e in el["cluster_ids"][1:]:
    cluster_id = Counter(e).most_common(1)[0][0]
    cluster_ids.append(cluster_id)
   bootstrap_cluster_ids.append(cluster_ids)

In [ ]:
scores = []
for a, b in zip(bootstrap_cluster_ids, bootstrap_cluster_ids_expected):
  s = adjusted_rand_score(a, b)
  scores.append(s)

print(np.mean(scores))

# Traditional clustering approaches

Let's use traditional clustering for comparison:

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
  words = nltk.word_tokenize(text.lower())
  words = [lemmatizer.lemmatize(word) for word in words if word.isalnum() and word not in stop_words]
  return " ".join(words)

texts = [s["text"] for s in submissions]
titles = [s["title"] for s in submissions]
processed_texts = [preprocess_text(t + "\n" + text) for t, text in zip(titles, texts)]

In [ ]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(processed_texts)

In [ ]:
import skfuzzy as fuzz
from sklearn.cluster import SpectralClustering, AgglomerativeClustering

def run_clusterizaion(data, n_clusters, algorithm="k-means", random_state=42):
  kwargs = {"random_state": random_state} if random_state else {}
  if algorithm == "fuzzy":
    if not isinstance(data, np.ndarray) and not isinstance(data, list):
      e_n = data.toarray()
    else:
      e_n = np.array(data)
    _, u, _, _, _, _, _ = fuzz.cmeans(
      e_n.T,
      n_clusters,
      m=2,
      error=0.005,
      maxiter=1000
    )
    labels = np.argmax(u, axis=0)
  elif algorithm == "k-means":
    cl = KMeans(n_clusters=n_clusters, n_init="auto", **kwargs)
    labels = cl.fit_predict(data)
  elif algorithm == "k-means++":
    cl = KMeans(n_clusters=n_clusters, init="k-means++", **kwargs)
    labels = cl.fit_predict(data)
  elif algorithm == "ahc":
    if not isinstance(data, np.ndarray) and not isinstance(data, list):
      data = data.toarray()
    cl = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
    labels = cl.fit_predict(data)
  else:
    cl = SpectralClustering(n_clusters=n_clusters, affinity='nearest_neighbors', n_neighbors=2, **kwargs)
    labels = cl.fit_predict(data)
  score = silhouette_score(data, labels)
  return labels, score

In [ ]:
def get_matrix(submissions):
  texts = [s["text"] for s in submissions]
  titles = [s["title"] for s in submissions]
  processed_texts = [preprocess_text(t + "\n" + text) for t, text in zip(titles, texts)]
  vectorizer = TfidfVectorizer()
  tfidf_matrix = vectorizer.fit_transform(processed_texts)
  return tfidf_matrix

In [ ]:
emb_model = VertexAIEmbeddings(
    model_name="gemini-embedding-001", project="kuligin-sandbox1")
embeddings = []

for title, text in zip(titles, texts):
  emb = emb_model.embed([f"{title}\n{text}"], embeddings_task_type="CLUSTERING")
  embeddings.append(emb)
  if len(embeddings) % 100 == 0:
    print(len(embeddings))

embeddings = [e[0] for e in embeddings]

In [ ]:
default_clusters = 26

algs = ["fuzzy", "k-means", "k-means++", "ahc", "spectral"]
data_types = ["tf-idf", "emb"]
clusters = {k: {k1: {}  for k1 in data_types} for k in algs}

possible_clusters = range(2, 26)
for alg in algs:
  for data_type in data_types:
    if data_type == "tf-idf":
      data = tfidf_matrix
    elif data_type == "emb":
      data = embeddings
    silhouette_scores = []
    for n_clusters in possible_clusters:
      _, score = run_clusterizaion(data, n_clusters, algorithm=alg)
      silhouette_scores.append(score)
    n_clusters = silhouette_scores.index(max(silhouette_scores))+2
    print(f"Optimal amount of clusters for {alg} and {data_type} is {n_clusters}")
    labels, score = run_clusterizaion(data, n_clusters, algorithm=alg)
    score_tf = silhouette_score(tfidf_matrix, labels)
    score_emb_gem = silhouette_score(embeddings, labels)
    print(score_tf, score_emb_gem)
    clusters[alg][data_type] = {
        "n_clusters": int(n_clusters), "labels": [int(e) for e in labels], "SC_tf": score_tf,
        "SC_emb_gem": score_emb_gem}

In [ ]:
clusters_df = {"alg": [], "data": [], "n_clusters": [], "SC_tf": [], "SC_emb_gem": []}
for k, v in clusters.items():
  for k1, v1 in v.items():
    clusters_df["alg"].append(k)
    clusters_df["data"].append(k1)
    for c in ["n_clusters", "SC_tf", "SC_emb_gem"]:
      clusters_df[c].append(v1[c])

clusters_df = pd.DataFrame(clusters_df)

In [ ]:
clusters_df.head(20)

## Interpetability

In [ ]:
with open(f"{file_name}_full_clusters.json", "w") as wf:
  wf.write(json.dumps(clusters))

In [ ]:
for alg in algs:
  for data_type in data_types:
    labels = clusters[alg][data_type]["labels"]
    cluster_descriptions, topics = [], []
    tasks = []
    for cluster_id in range(min(labels), max(labels)+1):
      submissions_slice = [s for s, l in zip(submissions, labels) if l == cluster_id]
      tasks.append(chain_cluster_description.ainvoke({"questions": format_submissions(submissions_slice), "disease": disease}))
      time.sleep(1)
    clusters_descriptions = await asyncio.gather(*tasks)
    tasks = [chain_cluster_header.ainvoke(s) for s in clusters_descriptions]
    topics = await asyncio.gather(*tasks)
    clusters[alg][data_type]["cluster_descriptions"] = clusters_descriptions
    clusters[alg][data_type]["cluster_topics"] = topics
    print(alg, data_type)

In [ ]:
with open(f"{file_name}_full_clusters.json", "w") as wf:
  wf.write(json.dumps(clusters))

In [ ]:
for k, v in clusters.items():
  for k1, v1 in v.items():
    v1["labels"] = [int(e) for e in v1["labels"]]

with open(f"{file_name}_full_clusters.json", "w") as wf:
  wf.write(json.dumps(clusters))

In [ ]:
with open(f"{file_name}_full_clusters.json", "r") as rf:
  r = json.loads(rf.read())

## Stability

In [ ]:
algs = ["fuzzy", "k-means", "k-means++", "ahc", "spectral"]
data_types = ["tf-idf", "emb"]


m = 20
for alg in algs:
  for data_type in data_types:
    if data_type == "tf-idf":
      data = get_matrix(submissions)
    elif data_type == "emb":
      data = embeddings
    n_clusters = clusters[alg][data_type]["n_clusters"]
    print(f"Running stability calculation for {alg} and {data_type}, n={n_clusters}")
    #if n_clusters < 4:
    #  n_clusters = default_clusters

    base_labels, _ = run_clusterizaion(data, n_clusters, algorithm=alg)

    bootstrap_indices = []
    bootstrap_raw_labels = []
    bootstrap_labels = []
    scores = []

    for i in range(m):
      sampled_submissions = random.choices(submissions, k=len(submissions))
      sampled_indices = []
      for s in sampled_submissions:
        for ind, s1 in enumerate(submissions):
          if s["text"] == s1["text"] and s["title"] == s1["title"]:
            sampled_indices.append(ind)
            continue

      bootstrap_indices.append(sampled_indices)
      if data_type == "tf-idf":
        data = get_matrix(sampled_submissions)
      elif data_type == "emb":
        data = list(np.array(embeddings)[sampled_indices])

      labels, _ = run_clusterizaion(data, n_clusters, algorithm=alg)
      bootstrap_ids = base_labels[sampled_indices]
      bootstrap_raw_labels.append(bootstrap_ids)
      bootstrap_labels.append(labels)

      s = adjusted_rand_score(labels, bootstrap_ids)
      print(s)
      scores.append(s)
    std_err = np.std(scores) / np.sqrt(m-1)
    print(f"Score = {np.mean(scores)}, {t_crit * std_err}")
    clusters[alg][data_type]["stability_score"] = np.mean(scores)
    clusters[alg][data_type]["stability_std_err"] = std_err

In [ ]:
import random

alg = algs[1]
print(alg)

n_clusters = 26
tfidf_matrix = get_matrix(submissions)
base_labels, _ = run_clusterizaion(tfidf_matrix, n_clusters, algorithm=alg)

bootstrap_indices = []
bootstrap_raw_labels = []
bootstrap_labels = []
scores = []

n = 20
for i in range(n):
  sampled_submissions = random.choices(submissions, k=len(submissions))
  sampled_indices = []
  for s in sampled_submissions:
    found = False
    for ind, s1 in enumerate(submissions):
      if s["text"] == s1["text"] and s["title"] == s1["title"]:
        sampled_indices.append(ind)
        found = True
        continue
    if not found:
      print(s)

  bootstrap_indices.append(sampled_indices)

  tfidf_matrix = get_matrix(sampled_submissions)
  labels, _ = run_clusterizaion(tfidf_matrix, n_clusters, algorithm=alg)

  bootstrap_ids = base_labels[sampled_indices]
  bootstrap_raw_labels.append(bootstrap_ids)
  bootstrap_labels.append(labels)

  s = adjusted_rand_score(labels, bootstrap_ids)
  print(s)
  scores.append(s)

std_err = np.std(scores) / np.sqrt(n-1)
print(f"Score = {np.mean(scores)}, {t_crit * std_err}")

In [ ]:
import random

alg = algs[4]
print(alg)

n_clusters = 21
base_labels, _ = run_clusterizaion(embeddings, n_clusters, algorithm=alg)

bootstrap_indices = []
bootstrap_raw_labels = []
bootstrap_labels = []
scores = []

for i in range(20):
  sampled_submissions = random.choices(submissions, k=len(submissions))
  sampled_indices = []
  for s in sampled_submissions:
    found = False
    for ind, s1 in enumerate(submissions):
      if s["text"] == s1["text"] and s["title"] == s1["title"]:
        sampled_indices.append(ind)
        found = True
        continue
    if not found:
      print(s)

  bootstrap_indices.append(sampled_indices)

  emb = list(np.array(embeddings)[sampled_indices])
  labels, _ = run_clusterizaion(emb, n_clusters, algorithm=alg)

  bootstrap_ids = base_labels[sampled_indices]
  bootstrap_raw_labels.append(bootstrap_ids)
  bootstrap_labels.append(labels)

  s = adjusted_rand_score(labels, bootstrap_ids)
  print(s)
  scores.append(s)

std_err = np.std(scores) / np.sqrt(n-1)
print(f"Score = {np.mean(scores)}, {t_crit * std_err}")

In [ ]:
plt.plot(range(2, 32), silhouette_scores)
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")
plt.title("K-means clustering (embeddings)")
plt.show()

In [ ]:
with open(f"{file_name}_full_clusters.json", "w") as wf:
  wf.write(json.dumps(clusters))

# LDA

In [ ]:
import gensim
import gensim.corpora as corpora
from gensim.models import CoherenceModel
from gensim.utils import simple_preprocess
from gensim.models.ldamodel import LdaModel

In [ ]:
def preprocess_data(documents):
 stop_words = stopwords.words("english")
 texts = [[word for word in simple_preprocess(str(doc)) if word not in stop_words] for doc in documents]
 return texts

In [ ]:
processed_texts_lda = preprocess_data(processed_texts)
id2word = corpora.Dictionary(processed_texts_lda)
corpus = [id2word.doc2bow(text) for text in processed_texts_lda]

In [ ]:
scores = []
for i in range(2,32):
  lda_model = LdaModel(corpus=corpus, id2word=id2word, num_topics=i, random_state=42, passes=10, alpha="auto", per_word_topics=True)
  coherence_model_lda = CoherenceModel(model=lda_model, texts=processed_texts_lda, dictionary=id2word, coherence="c_v")
  coherence_lda = coherence_model_lda.get_coherence()
  print("Coherence Score: ", coherence_lda)
  scores.append(coherence_lda)

n_clusters = scores.index(max(scores))+2
print(f"Optimal amount of clusters {n_clusters}")

In [ ]:
def get_document_topic_distribution(lda_model, corpus):
    topic_distributions = []
    for doc in lda_model[corpus]:
        topic_distributions.append(doc[0])
    return topic_distributions


def get_document_cluster_labels(topic_distributions):
    labels = []
    for doc_distribution in topic_distributions:
        dominant_topic = max(doc_distribution, key=lambda item: item[1])[0]
        labels.append(dominant_topic)
    return labels



In [ ]:
import random

n_clusters = 21

texts = [s["text"] for s in submissions]
titles = [s["title"] for s in submissions]
processed_texts = [preprocess_text(t + "\n" + text) for t, text in zip(titles, texts)]
processed_texts_lda = preprocess_data(processed_texts)
id2word = corpora.Dictionary(processed_texts_lda)
corpus = [id2word.doc2bow(text) for text in processed_texts_lda]

lda_model = LdaModel(corpus=corpus, id2word=id2word, num_topics=n_clusters, passes=10, alpha="auto", per_word_topics=True)
topic_distributions = get_document_topic_distribution(lda_model, corpus)
base_labels = get_document_cluster_labels(topic_distributions)

bootstrap_indices = []
bootstrap_raw_labels = []
bootstrap_labels = []
scores = []

for i in range(20):
  sampled_submissions = random.choices(submissions, k=len(submissions))
  sampled_indices = []
  for s in sampled_submissions:
    found = False
    for ind, s1 in enumerate(submissions):
      if s["text"] == s1["text"] and s["title"] == s1["title"]:
        sampled_indices.append(ind)
        found = True
        continue
    if not found:
      print(s)

  bootstrap_indices.append(sampled_indices)

  texts = [s["text"] for s in sampled_submissions]
  titles = [s["title"] for s in sampled_submissions]
  processed_texts = [preprocess_text(t + "\n" + text) for t, text in zip(titles, texts)]
  processed_texts_lda = preprocess_data(processed_texts)
  id2word = corpora.Dictionary(processed_texts_lda)
  corpus = [id2word.doc2bow(text) for text in processed_texts_lda]

  lda_model = LdaModel(corpus=corpus, id2word=id2word, num_topics=n_clusters, passes=10, alpha="auto", per_word_topics=True)
  topic_distributions = get_document_topic_distribution(lda_model, corpus)
  labels = get_document_cluster_labels(topic_distributions)

  bootstrap_ids = list(np.array(base_labels)[sampled_indices])
  bootstrap_raw_labels.append(bootstrap_ids)
  bootstrap_labels.append(labels)

  s = adjusted_rand_score(labels, bootstrap_ids)
  print(s)
  scores.append(s)

std_err = np.std(scores) / np.sqrt(n-1)
print(f"Score = {np.mean(scores)}, {t_crit * std_err}")

In [ ]:
cluster_descriptions, topics = [], []
for cluster_id in range(min(labels), max(labels)+1):
    submissions_slice = [s for s, l in zip(submissions, labels) if l == cluster_id]
    summary = chain_cluster_description.invoke({"questions": format_submissions(submissions_slice)})
    cluster_descriptions.append(summary)
    topic = chain_cluster_description.invoke(summary)
    topics.append(topic)